# Soup GRPO task: step 3, control run (shuffled rewards) + base model evaluation (Kaggle T4)

**Before running:** Settings, then **Accelerator: GPU T4 x2** and **Internet: On**. Then **Save Version ->
Save & Run All (Commit)** and close the tab. Takes about **6.5 hours**. It can run at the same time as the other
training notebook. When the version shows Complete, open its **Output** tab and download **`soup_results.zip`**.

Configuration and the SHIP rule are pre-registered in `predictions/PREDICTIONS.md` (training setup, the
amendment after dry run 1, and "Evaluation protocol and the SHIP rule"). Steps 1-5 are the usual safety checks.
6. **Control training: 40 steps with shuffled rewards** (~4.2 h): same everything, but the reward a sequence gets is another sequence's, so no real signal. Any gain here comes from the process, not from learning the task.
7. **Held-out evaluation** of the control adapter.
8. **KL** of the control policy to the base model.
9. **Held-out evaluation of the base model** (the reference for all comparisons).
10. **GSM8K** for the base model.
If a session ever dies mid-run: add this notebook's failed version as an **Input** (Add Input -> Your Work) and
run again; the training cell finds the saved run directory and resumes exactly (H19).

In [ ]:
# 1. Unpack code + data (git e0cd25a+dirty, bundle sha256 213cedd4ebee95c8)
import base64, hashlib, io, json, os, subprocess, sys, tarfile, time
from pathlib import Path
LOCAL_TEST = os.environ.get("SOUP_LOCAL_TEST") == "1"   # only for testing this notebook on a laptop
WORK = Path(os.environ.get("SOUP_WORK", "/kaggle/working"))
WORK.mkdir(parents=True, exist_ok=True)
BUNDLE = "..."
raw = base64.b64decode(BUNDLE)
assert hashlib.sha256(raw).hexdigest() == "213cedd4ebee95c8193a2017d397f0980cc140454c5879367d23c0639363d4f6", "bundle corrupted"
with tarfile.open(fileobj=io.BytesIO(raw), mode="r:gz") as tar:
    tar.extractall(WORK, **({"filter": "data"} if hasattr(tarfile, "data_filter") else {}))
REPO = WORK / "soup-grpo"
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
man = json.loads((REPO / "data/MANIFEST.json").read_text())
for f, info in man["files"].items():
    assert hashlib.sha256((REPO / "data" / f).read_bytes()).hexdigest() == info["sha256"], f
(REPO / "results/logs").mkdir(parents=True, exist_ok=True)
(REPO / "results/pilot").mkdir(parents=True, exist_ok=True)
print("code at", REPO, "| data files verified:", len(man["files"]), "| bundle git rev e0cd25a+dirty")

In [ ]:
# helper: run a command, stream its output live, keep a copy in results/logs
def run(cmd, log=None, check=True, env=None):
    print("\n$", " ".join(map(str, cmd)), flush=True)
    t0 = time.time()
    p = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, **(env or {})})
    lines = []
    for line in p.stdout:
        if "Warning" in line or "warn(" in line:
            continue
        print(line, end="", flush=True)
        lines.append(line)
    p.wait()
    if log:
        (REPO / "results/logs" / log).write_text("".join(lines))
    print(f"[exit {p.returncode}, {time.time() - t0:.0f}s]", flush=True)
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed: {cmd}")
    return p.returncode

PY = sys.executable
STOP = None   # set by the first failing step; later steps are skipped, results are still packed
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # one GPU, as in the task
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # less cache fragmentation (pilot 1: 2 GiB)
MODEL = os.environ.get("SOUP_MODEL") or ("Qwen/Qwen3-0.6B" if LOCAL_TEST else "Qwen/Qwen3-4B")   # SOUP_MODEL: local tests only
DEVICE = "cpu" if LOCAL_TEST else "cuda"

In [ ]:
# 2. Environment + dependencies
if not LOCAL_TEST:
    run([PY, "-m", "pip", "install", "-q", "transformers>=4.51", "safetensors>=0.4", "huggingface_hub"], check=False)
    TS = time.strftime("%Y%m%d-%H%M%S")
    env_txt = subprocess.run("nvidia-smi; nvidia-smi --query-gpu=name,memory.total,driver_version,"
                             "pcie.link.gen.current,pcie.link.gen.max,pcie.link.width.current,pcie.link.width.max "
                             "--format=csv; free -g; nproc; df -h /kaggle/working",
                             shell=True, capture_output=True, text=True).stdout
    import torch, transformers
    env_txt += f"\ntorch {torch.__version__} cuda {torch.version.cuda} transformers {transformers.__version__}"
    env_txt += f"\ndevice {torch.cuda.get_device_name(0)} capability {torch.cuda.get_device_capability(0)}"
    env_txt += f" bf16_supported {torch.cuda.is_bf16_supported()}"
    (REPO / f"results/logs/env_{TS}.txt").write_text(env_txt)
    print(env_txt)

In [ ]:
# 3. Session monitor (nvidia-smi + host RAM every second) and the 8 GB ballast
if STOP is None:
    try:
        if not LOCAL_TEST:
            TS = time.strftime("%Y%m%d-%H%M%S")
            smi_log = open(REPO / f"results/logs/nvsmi_{TS}.csv", "w")
            monitor = subprocess.Popen(["nvidia-smi", "--query-gpu=timestamp,memory.used,memory.total,utilization.gpu,"
                                        "pcie.link.gen.current,pcie.link.width.current,temperature.gpu,power.draw,clocks.sm",
                                        "--format=csv", "-l", "1"], stdout=smi_log, stderr=subprocess.STDOUT)
            host_mon = subprocess.Popen(["bash", "-c", f"while true; do echo \"$(date -Is),$(free -m | awk '/Mem:/ {{print $3\",\"$7}}')\""
                                         f" >> {REPO}/results/logs/hostmem_{TS}.csv; sleep 1; done"])
            ballast = subprocess.Popen([PY, "scripts/gpu/ballast.py", "--cap-mib", "8192"],
                                       stdout=open(REPO / "results/logs/ballast.out", "w"), stderr=subprocess.STDOUT)
            for _ in range(120):
                if (REPO / "results/logs/ballast.json").exists():
                    break
                time.sleep(1)
            BALLAST = json.loads((REPO / "results/logs/ballast.json").read_text())
            print("ballast:", BALLAST)
            assert BALLAST["free_left_mib"] <= 8192, "cap not in place"
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 4. Model download (~8 GB) and correctness of the engine on this GPU
if STOP is None:
    try:
        t0 = time.time()
        from soupgrpo.runtime import resolve_model_dir
        print("downloading", MODEL, "(~8 GB, a few minutes; no output until done) ...", flush=True)
        print("model at", resolve_model_dir(MODEL), f"({time.time() - t0:.0f}s)")
        run([PY, "-m", "pytest", "-q", "-x"], log="pytest.txt", check=False)
        if not LOCAL_TEST:
            run([PY, "scripts/gpu/cuda_equivalence.py"], log="cuda_equivalence.txt")
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 5. PCIe bandwidth (pinned vs pageable, one 4B layer = 202 MB) + CUDA context cost
if STOP is None:
    try:
        if not LOCAL_TEST:
            run([PY, "scripts/gpu/pcie_bench.py"], log="pcie.txt")
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 6. GRPO training, run "control" (configuration pre-registered; control differs only by --shuffle-rewards)
if STOP is None:
    try:
        import shutil
        NAME = "control"
        RUN = REPO / "runs" / NAME
        # resume support: a previous (failed) version of this notebook attached as Input
        prev = sorted(Path("/kaggle/input").glob(f"**/runs/{NAME}/state.json")) if Path("/kaggle/input").exists() else []
        if prev and not (RUN / "state.json").exists():
            shutil.copytree(prev[-1].parent, RUN)
            print("RESUMING from", prev[-1].parent)
        if LOCAL_TEST:
            STEPS = 2
            cfg = ["--prompts-per-step", "2", "--group", "4", "--max-new", "32", "--gen-batch", "8",
                   "--train-seqs-per-pass", "8", "--micro-batch", "4", "--dtype", "fp32"]
        else:
            STEPS = 40
            cfg = ["--prompts-per-step", "6", "--group", "8", "--max-new", "512", "--gen-batch", "48",
                   "--train-seqs-per-pass", "24", "--micro-batch", "8", "--dtype", "fp16"]
        run([PY, "scripts/train_grpo.py", "--model", MODEL, "--data", "data/train_n5_main.jsonl", "--prompt", "v2",
             "--steps", STEPS, "--lr", "2e-5", "--warmup", "2", "--beta", "0", "--log-kl-every", "5", "--seed", "1234",
             "--temperature", "1.0", *cfg, *["--shuffle-rewards"], "--run-dir", RUN], log=f"train_{NAME}.txt")
        M = [json.loads(l) for l in open(RUN / "metrics.jsonl")]
        assert len(M) == STEPS and [m["step"] for m in M] == list(range(STEPS)), "steps missing or duplicated"
        ADAPTER = RUN / "ckpt" / f"step_{STEPS:05d}"
        assert (ADAPTER / "adapter_model.safetensors").exists(), ADAPTER
        shutil.copytree(ADAPTER, REPO / f"results/adapters/{NAME}", ignore=shutil.ignore_patterns("optimizer.pt", "gen_state.pt"),
                        dirs_exist_ok=True)
        shutil.copytree(RUN, REPO / f"results/runs/{NAME}", ignore=shutil.ignore_patterns("ckpt", ".lock"), dirs_exist_ok=True)
        import numpy as np
        for lo in range(0, STEPS, 10):
            w = M[lo:lo + 10]
            print(f"steps {lo}-{lo + len(w) - 1}: reward {np.mean([m['reward'] for m in w]):.3f} "
                  f"trunc {np.mean([m['truncated_share'] for m in w]):.2f} zero-adv {np.mean([m['zero_adv_group_share'] for m in w]):.2f} "
                  f"len {np.mean([m['len_mean'] for m in w]):.0f} adapter {w[-1]['adapter_delta_norm']:.3f} "
                  f"step {np.mean([m['time_s']['total'] for m in w]):.0f}s")
        print("KL to base (logged steps):", [(m["step"], m["kl_mean"]) for m in M if m.get("kl_mean") is not None])
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 7. Held-out evaluation of the "control" adapter: 500 new 5-number puzzles, greedy, 512 tokens
if STOP is None:
    try:
        run([PY, "scripts/sample_eval.py", "--model", MODEL, "--adapter", ADAPTER, "--device", DEVICE,
             "--dtype", "fp32" if LOCAL_TEST else "fp16", "--prompt", "v2", "--data", "data/heldout_n5.jsonl",
             "--samples", "1", "--temperature", "0", "--max-new", "32" if LOCAL_TEST else "512", "--batch", "48",
             *(["--limit", "6"] if LOCAL_TEST else []), "--out", REPO / "results/eval/control"], log="eval_control.txt")
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 8. H28: exact KL(policy || base) on the first 200 held-out answers of this adapter
if STOP is None:
    try:
        run([PY, "scripts/kl_eval.py", "--model", MODEL, "--adapter", ADAPTER, "--eval-jsonl", REPO / "results/eval/control.jsonl",
             "--n", "6" if LOCAL_TEST else "200", "--dtype", "fp32" if LOCAL_TEST else "fp16",
             "--out", REPO / "results/eval/kl_control.json"], log="kl_control.txt")
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 9. Held-out evaluation of the BASE model (reference for every comparison), same protocol
if STOP is None:
    try:
        run([PY, "scripts/sample_eval.py", "--model", MODEL, "--device", DEVICE,
             "--dtype", "fp32" if LOCAL_TEST else "fp16", "--prompt", "v2", "--data", "data/heldout_n5.jsonl",
             "--samples", "1", "--temperature", "0", "--max-new", "32" if LOCAL_TEST else "512", "--batch", "48",
             *(["--limit", "6"] if LOCAL_TEST else []), "--out", REPO / "results/eval/base"], log="eval_base.txt")
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 10. H10: GSM8K, first 100 test questions, greedy, 512 tokens (base)
if STOP is None:
    try:
        run([PY, "scripts/gsm8k_eval.py", "--model", MODEL, *[], "--n", "4" if LOCAL_TEST else "100",
             "--max-new", "16" if LOCAL_TEST else "512", "--batch", "48", "--dtype", "fp32" if LOCAL_TEST else "fp16",
             "--out", REPO / "results/eval/gsm8k_base"], log="gsm8k_base.txt")
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 9. Peak memory of THIS run (nvidia-smi used minus the ballast) and pack results (always runs)
if not LOCAL_TEST and "smi_log" in globals() and "BALLAST" in globals():
    import csv
    rows = list(csv.reader(open(smi_log.name)))[1:]
    used = [int(r[1].strip().split()[0]) for r in rows if len(r) > 1 and r[1].strip().split()[0].isdigit()]
    # subtract everything that was on the card before our processes started (ballast + driver baseline);
    # pilot 1 subtracted only the ballast and over-reported by the 107 MiB baseline (E12)
    peak = max(used) - BALLAST["smi_used_after_mib"]
    cap = {"smi_peak_total_mib": max(used), "smi_before_us_mib": BALLAST["smi_used_after_mib"],
           "ballast_used_mib": BALLAST["ballast_used_mib"], "our_peak_mib": peak,
           "cap_mib": BALLAST["cap_mib"], "under_cap": peak <= BALLAST["cap_mib"], "samples": len(used)}
    (REPO / "results/logs/cap_summary.json").write_text(json.dumps(cap, indent=2))
    print(cap)
import shutil
(REPO / "results/logs/stop_reason.txt").write_text(str(STOP))
print("STOP reason:", STOP)
zip_base = WORK / "soup_results"
shutil.make_archive(str(zip_base), "zip", REPO / "results")
print("DOWNLOAD:", str(zip_base) + ".zip")